# Adversarial Training From Split CSVs

This notebook trains the adversarial Model B loop directly from prebuilt `train`, `test`, and `mutated_data` split CSVs.

- It keeps the original Python scripts unchanged.
- `train_split.csv` is used for adversarial training.
- `test_split.csv` and `mutated_data_split.csv` are used for evaluation after training.


In [1]:
%pip install numpy pandas scikit-learn tqdm matplotlib seaborn flask jupyter ipywidgets \
transformers huggingface_hub datasets accelerate sentence-transformers nltk sentencepiece faiss-cpu

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import sys

import torch

from distilbert_experiment import (
    build_detector_for_model_b,
    load_records_csv,
    records_to_prompt_lists,
    run_single_experiment,
    split_by_label,
)
from train_loop import AdversarialTrainer, TrainingConfig


print(f"Torch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")


Torch version: 2.8.0+cu128
CUDA available: True
CUDA device: NVIDIA L4


In [3]:
from pathlib import Path

TRAIN_SPLIT_CSV = Path("train_split.csv")
TEST_SPLIT_CSV = Path("test_split.csv")
MUTATED_SPLIT_CSV = Path("mutated_data_split.csv")

NOTEBOOK_OUTPUT_DIR = Path("notebook_adversarial_training")
NOTEBOOK_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [TRAIN_SPLIT_CSV, TEST_SPLIT_CSV, MUTATED_SPLIT_CSV]:
    print(path.resolve())
    if not path.exists():
        raise FileNotFoundError(f"Missing split CSV: {path}")


/home/mvibhute/mvibhute/results/train_split.csv
/home/mvibhute/mvibhute/results/test_split.csv
/home/mvibhute/mvibhute/results/mutated_data_split.csv


In [4]:
train_records = load_records_csv(TRAIN_SPLIT_CSV)
test_records = load_records_csv(TEST_SPLIT_CSV)
mutated_records = load_records_csv(MUTATED_SPLIT_CSV)

train_jailbreak_prompts, train_benign_prompts = split_by_label(train_records)
test_prompts, test_labels = records_to_prompt_lists(test_records)

summary = {
    "train_rows": len(train_records),
    "train_jailbreak": len(train_jailbreak_prompts),
    "train_benign": len(train_benign_prompts),
    "test_rows": len(test_records),
    "mutated_rows": len(mutated_records),
}
summary


{'train_rows': 19229,
 'train_jailbreak': 8152,
 'train_benign': 11077,
 'test_rows': 4808,
 'mutated_rows': 9012}

## Training Config

The defaults below are GPU-friendly but still conservative. A good first run is `num_rounds=1` with the faster mutator strategies. Once that works on your machine, you can increase rounds, strategies, or variants.


In [5]:
config = TrainingConfig(
    model_name="distilbert/distilbert-base-uncased",
    num_epochs=3,
    batch_size=16,
    learning_rate=2e-5,
    max_length=512,
    warmup_ratio=0.1,
    weight_decay=0.01,
    num_rounds=4,
    variants_per_seed=3,
    min_fool_confidence=0.4,
    detector_model_threshold=0.5,
    mutator_strategies=["wordnet", "bert", "t5", "backtranslate", "roleplay", "structural"],
    cache_threshold=0.72,
    output_dir=str(NOTEBOOK_OUTPUT_DIR / "model_b_rounds"),
    cache_save_path=str(NOTEBOOK_OUTPUT_DIR / "model_b_cache"),
    history_save_path=str(NOTEBOOK_OUTPUT_DIR / "model_b_history.json"),
    resume_state_path=str(NOTEBOOK_OUTPUT_DIR / "model_b_rounds" / "resume_state.json"),
    seed=42,
    val_split=0.15,
    verbose=True,
    show_progress_bars=True,
    dry_run=False,
)

config


TrainingConfig(model_name='distilbert/distilbert-base-uncased', num_epochs=3, batch_size=16, learning_rate=2e-05, max_length=512, warmup_ratio=0.1, weight_decay=0.01, num_rounds=4, variants_per_seed=3, min_fool_confidence=0.4, detector_model_threshold=0.5, mutator_strategies=['wordnet', 'bert', 't5', 'backtranslate', 'roleplay', 'structural'], cache_threshold=0.72, output_dir='notebook_adversarial_training/model_b_rounds', cache_save_path='notebook_adversarial_training/model_b_cache', history_save_path='notebook_adversarial_training/model_b_history.json', resume_state_path='notebook_adversarial_training/model_b_rounds/resume_state.json', seed=42, val_split=0.15, test_split=0.1, verbose=True, show_progress_bars=True, sample_preview_count=3, dry_run=False)

In [6]:
# Main training cell.
# This is the long-running step.

trainer = AdversarialTrainer(config)
final_model, final_cache = trainer.run(
    jailbreak_prompts=train_jailbreak_prompts,
    benign_prompts=train_benign_prompts,
    test_prompts=test_prompts,
    test_labels=test_labels,
)

model_b_path = trainer.final_checkpoint_path or trainer.current_model_source
model_b_cache_path = trainer.final_cache_path or config.cache_save_path

print("Model path:", model_b_path)
print("Cache path:", model_b_cache_path)
print("History path:", config.history_save_path)


[20:22:22] [trainer] Device: cuda
[JailbreakMutator] Loaded strategies: ['wordnet', 'bert', 't5', 'backtranslate', 'roleplay', 'structural']
[JailbreakMutator] Mode: independent
[20:22:22] [trainer] Starting from model source: distilbert/distilbert-base-uncased
[20:22:22] [trainer] Loading tokenizer: distilbert/distilbert-base-uncased


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /home/mvibhute/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache/faiss] Built index: 8152 prompts, dim=384, model=sentence-transformers/all-MiniLM-L6-v2


...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /home/mvibhute/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache/faiss] Loaded 9522 prompts from notebook_adversarial_training/model_b_cache.faiss
[20:22:34] [resume] Adversarial training was already completed earlier. Loading the saved final checkpoint.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model path: notebook_adversarial_training/model_b_rounds/round_4
Cache path: notebook_adversarial_training/model_b_cache
History path: notebook_adversarial_training/model_b_history.json


## Evaluate The Trained Adversarial Model

This section reuses the saved checkpoint and cache, then scores the held-out `test` and `mutated_data` splits.

In [7]:
test_csv = NOTEBOOK_OUTPUT_DIR / "test_model_b_on_test.csv"
mutated_csv = NOTEBOOK_OUTPUT_DIR / "test_model_b_on_mutated_data.csv"

detector_test = build_detector_for_model_b(
    model_path=model_b_path,
    cache_path=str(model_b_cache_path),
    cache_threshold=config.cache_threshold,
    model_threshold=config.detector_model_threshold,
)
summary_test = run_single_experiment(
    experiment_name="notebook_model_b_on_test",
    model_variant="Model B",
    dataset_variant="test",
    detector=detector_test,
    records=test_records,
    csv_path=test_csv,
)

detector_mutated = build_detector_for_model_b(
    model_path=model_b_path,
    cache_path=str(model_b_cache_path),
    cache_threshold=config.cache_threshold,
    model_threshold=config.detector_model_threshold,
)
summary_mutated = run_single_experiment(
    experiment_name="notebook_model_b_on_mutated_data",
    model_variant="Model B",
    dataset_variant="mutated_data",
    detector=detector_mutated,
    records=mutated_records,
    csv_path=mutated_csv,
)

evaluation_summary = {
    "test": summary_test,
    "mutated_data": summary_mutated,
}

summary_path = NOTEBOOK_OUTPUT_DIR / "evaluation_summary.json"
summary_path.write_text(json.dumps(evaluation_summary, indent=2), encoding="utf-8")
evaluation_summary


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /home/mvibhute/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache/faiss] Loaded 9522 prompts from notebook_adversarial_training/model_b_cache.faiss
[neural] Loading model from 'notebook_adversarial_training/model_b_rounds/round_4' on cuda...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

BertModel LOAD REPORT from: /home/mvibhute/.cache/huggingface/hub/models--sentence-transformers--all-MiniLM-L6-v2/snapshots/c9745ed1d9f207416be6d2e6f8de32d1f16199bf
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[cache/faiss] Loaded 9522 prompts from notebook_adversarial_training/model_b_cache.faiss
[neural] Loading model from 'notebook_adversarial_training/model_b_rounds/round_4' on cuda...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[neural] Model loaded.


{'test': {'num_examples': 4808,
  'accuracy': 0.8966,
  'precision': 0.8223,
  'recall': 0.9647,
  'f1': 0.8878,
  'avg_latency_ms': 4.998,
  'avg_confidence': 0.4974,
  'avg_jailbreak_confidence': 0.9638,
  'cache_hits': 1507,
  'cache_hit_rate': 0.3134,
  'tp': 1966,
  'fp': 425,
  'fn': 72,
  'tn': 2345,
  'experiment_name': 'notebook_model_b_on_test',
  'model_variant': 'Model B',
  'dataset_variant': 'test',
  'csv_path': 'notebook_adversarial_training/test_model_b_on_test.csv'},
 'mutated_data': {'num_examples': 9012,
  'accuracy': 0.4929,
  'precision': 0.488,
  'recall': 1.0,
  'f1': 0.6559,
  'avg_latency_ms': 3.734,
  'avg_confidence': 0.9904,
  'avg_jailbreak_confidence': 1.0,
  'cache_hits': 656,
  'cache_hit_rate': 0.0728,
  'tp': 4356,
  'fp': 4570,
  'fn': 0,
  'tn': 86,
  'experiment_name': 'notebook_model_b_on_mutated_data',
  'model_variant': 'Model B',
  'dataset_variant': 'mutated_data',
  'csv_path': 'notebook_adversarial_training/test_model_b_on_mutated_data.csv'}

In [8]:
history_path = Path(config.history_save_path)
if history_path.exists():
    history = json.loads(history_path.read_text(encoding="utf-8"))
    print(f"Loaded {len(history)} round records from {history_path}")
    history
else:
    print(f"History file not found yet: {history_path}")


Loaded 4 round records from notebook_adversarial_training/model_b_history.json
